In [3]:
import sys, os, time, math
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE

# ── sys.path (tham khảo notebook haberman) ─────────────────────
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
print(f"sys.path += {parent_dir}")

# ── Khai báo dataset & thư mục chạy (tham khảo notebook haberman) ──
DATASET_NAME = "haberman"
DATASET_FILE = "../Processing_Data/dataset/haberman.csv"

_DS_BASE = os.path.splitext(os.path.basename(DATASET_FILE))[0]
_RUN_TS = datetime.now().strftime("%d%m%Y_%H%M%S")
RUN_DIR = os.path.join("./Experiment", f"Test_{_DS_BASE}_{_RUN_TS}")
LOG_DIR = os.path.join(RUN_DIR, "logs")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"✔ RUN_DIR: {RUN_DIR}")
print(f"✔ LOG_DIR: {LOG_DIR}")

# IMPORT MODULES GỐC CỦA BẠN (sau khi đã set sys.path)
from wsvm.application import Wsvm
from svm.application import Svm
from fuzzy.weight import fuzzy

# 1. HÀM TIỆN ÍCH (Gmean, is_tomek, fuzzy_weight, lfb)
def Gmean(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred, labels=[-1.0, 1.0])
    TN, FP, FN, TP = cm.ravel()
    se = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    sp = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    return sp, se, math.sqrt(se * sp)

def is_tomek(X, y, class_type):
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    _, nbr = nn.kneighbors(X)
    nn_idx = nbr[:, 1]
    links = np.zeros(len(y), dtype=bool)
    class_excluded = [c for c in np.unique(y) if c not in class_type]
    X_dangxet, X_tl = [], []
    for i, target in enumerate(y):
        if target in class_excluded: continue
        if y[nn_idx[i]] != target and nn_idx[nn_idx[i]] == i:
            X_tl.append(i); X_dangxet.append(nn_idx[i]); links[i] = True
    return links, X_dangxet, X_tl

def compute_weight(X, y, beta=0.8):
    method, function = fuzzy.method(), fuzzy.function()
    pos_index, neg_index = np.where(y == 1), np.where(y == -1)
    d_own, d_opp, d_tam = method.distance_center_own_opposite_tam(X, y)
    W = function.func_own_opp_new(d_own, d_opp, pos_index, neg_index, d_tam)
    r_neg = len(pos_index) / max(len(neg_index), 1)
    m = np.zeros(len(y))
    m[pos_index] = W[pos_index] * 1.0
    m[neg_index] = W[neg_index] * r_neg
    return m

def data_tomelinks_parametric(C, ind_posX, ind_negX, weight, X_test, y_test, X_train, y_train, K_neighbors, s1, s2, s3, s4):
    new_W = np.copy(weight)
    clf = Wsvm(C, new_W)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    _, se, gm = Gmean(y_test, y_pred)
    if np.any(np.isnan(y_pred)): return new_W, 0.0, 0.0
    
    nn2 = NearestNeighbors(n_neighbors=K_neighbors).fit(X_train)
    all_train_preds = clf.predict(X_train)
    
    for ind, i in enumerate(ind_posX):
        j_idx = ind_negX[ind]
        if all_train_preds[i] == 1.0 and all_train_preds[j_idx] == 1.0:
            new_W[i] *= (1.0 + s1); new_W[j_idx] *= (1.0 - s1)
            knn_j = nn2.kneighbors(X_train[j_idx:j_idx+1])[1].flatten()
            if np.sum(all_train_preds[knn_j] == 1.0) > K_neighbors / 2: new_W[j_idx] *= s2
        elif all_train_preds[i] == -1.0 and all_train_preds[j_idx] == -1.0:
            new_W[i] *= (1.0 + s3); new_W[j_idx] *= (1.0 - s3)
            knn_i = nn2.kneighbors(X_train[i:i+1])[1].flatten()
            if np.sum(all_train_preds[knn_i] == -1.0) > K_neighbors / 2: new_W[i] *= s4
    return new_W, gm, se

def lfb_pso(C, ind_posX, ind_negX, weight, T, X_test, y_test, X_train, y_train, K, s1, s2, s3, s4):
    gmax, best_w, cur_w = 0.0, np.copy(weight), np.copy(weight)
    for _ in range(T):
        cur_w, gm, _ = data_tomelinks_parametric(C, ind_posX, ind_negX, cur_w, X_test, y_test, X_train, y_train, K, s1, s2, s3, s4)
        if gm > gmax: gmax, best_w = gm, np.copy(cur_w)
    return best_w, gmax

# =====================================================================
# 2. ĐỘNG CƠ ADVANCED PSO (LDIW + TVAC + SEEDING) CHUẨN MỰC
# =====================================================================
class ADVANCED_PSO_AFWCIL:
    def __init__(self, num_particles, iters, bounds, C, T, X_train, y_train, X_test, y_test):
        self.num_particles, self.iters, self.bounds = num_particles, iters, bounds
        self.C, self.T = C, T
        self.X_train, self.y_train, self.X_test, self.y_test = X_train, y_train, X_test, y_test
        _, self.ind_posX, self.ind_negX = is_tomek(X_train, y_train, class_type=[-1.0])
        self.init_weight = compute_weight(X_train, y_train)
        
    def fitness(self, particle):
        K = int(np.rint(float(np.asarray(particle).reshape(-1)[0])))
        s1, s2, s3, s4 = particle[1], particle[2], particle[3], particle[4]
        _, gmax = lfb_pso(self.C, self.ind_posX, self.ind_negX, self.init_weight, self.T, 
                          self.X_test, self.y_test, self.X_train, self.y_train, K, s1, s2, s3, s4)
        return gmax

    def optimize(self):
        dim = len(self.bounds)
        pos = np.random.rand(self.num_particles, dim)
        for d in range(dim):
            low, high = self.bounds[d]
            pos[:, d] = pos[:, d] * (high - low) + low
        
        # [CẢI TIẾN 1]: SEEDING - Gán hạt số 0 là thông số của tác giả gốc
        pos[0] = np.array([5, 0.1, 0.5, 0.1, 0.5], dtype=float)
        
        vel = np.zeros_like(pos)
        pbest_pos = np.copy(pos)
        pbest_val = np.array([self.fitness(p) for p in pos])
        
        gbest_idx = np.argmax(pbest_val)
        gbest_pos = np.copy(pbest_pos[gbest_idx])
        gbest_val = pbest_val[gbest_idx]
        
        print(f"👉 Khởi tạo ban đầu | G-mean Tốt nhất (thường là Hạt Seed): {gbest_val:.4f}")

        # [CẢI TIẾN 2 & 3]: LDIW và TVAC CHUẨN MỰC
        w_max, w_min = 0.9, 0.4
        c1_max, c1_min = 2.5, 0.5
        c2_max, c2_min = 2.5, 0.5  # Khai báo c2_max và c2_min cho chuẩn

        for i in range(self.iters):
            # Tính toán tỷ lệ tiến trình
            ratio = i / self.iters
            
            # w giảm dần từ 0.9 -> 0.4
            w = w_max - (w_max - w_min) * ratio
            
            # c1 GIẢM dần (Bớt tự tin cá nhân đi)
            c1 = c1_max - (c1_max - c1_min) * ratio
            
            # c2 TĂNG dần (Tăng sự đoàn kết bầy đàn lên)
            # CHÚ Ý DẤU CỘNG Ở ĐÂY
            c2 = c2_min + (c2_max - c2_min) * ratio 
            
            for j in range(self.num_particles):
                r1, r2 = np.random.rand(dim), np.random.rand(dim)
                vel[j] = w * vel[j] + c1 * r1 * (pbest_pos[j] - pos[j]) + c2 * r2 * (gbest_pos - pos[j])
                # Clipping
                for d in range(dim):
                    pos[j, d] = np.clip(pos[j, d], self.bounds[d][0], self.bounds[d][1])
                
                fit_val = self.fitness(pos[j])
                if fit_val > pbest_val[j]:
                    pbest_val[j], pbest_pos[j] = fit_val, np.copy(pos[j])
                if fit_val > gbest_val:
                    gbest_val, gbest_pos = fit_val, np.copy(pos[j])
            
            gbest_k = int(np.rint(float(gbest_pos[0])))
            print(f"Iteration {i+1:02d}/{self.iters} | w={w:.2f}, c1={c1:.2f}, c2={c2:.2f} | GBest: {gbest_val:.4f} | " 
                  f"Params: K={gbest_k}, s1={gbest_pos[1]:.2f}, s2={gbest_pos[2]:.2f}, s3={gbest_pos[3]:.2f}, s4={gbest_pos[4]:.2f}")
            
        return gbest_pos, gbest_val

# =====================================================================
# 3. KỊCH BẢN CHẠY TEST ĐÚNG 1 FOLD
# =====================================================================
print("\n" + "="*60)
print("BẮT ĐẦU TEST NHANH 1 FOLD VỚI PHƯƠNG PHÁP TỐI ƯU TOÀN DIỆN")
print("=")

# Load data từ biến khai báo phía trên
df = pd.read_csv(DATASET_FILE)
df['class'] = df['class'].map({2: 1.0, 1: -1.0})
y_all = np.asarray(df['class'].to_numpy(), dtype=float)
X_all = np.asarray(df.drop(columns=['class']).to_numpy(), dtype=float)

# Cắt ĐÚNG 1 FOLD (Giữ nguyên random_state để cố định test set)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(skf.split(X_all, y_all))
scaler = StandardScaler().fit(X_all[train_idx])
X_tr, X_te = scaler.transform(X_all[train_idx]), scaler.transform(X_all[test_idx])
y_tr, y_te = y_all[train_idx], y_all[test_idx]

C_val = 100
T_val = 5
print("\n1. ĐÁNH GIÁ BASELINE (W-SVM):")
base_clf = Wsvm(C_val, np.ones(len(y_tr)))
base_clf.fit(X_tr, y_tr)
pred_base = base_clf.predict(X_te)
_, _, gm_base = Gmean(y_te, pred_base)
print(f" -> G-mean W-SVM: {gm_base:.4f}")

print("\n2. ĐÁNH GIÁ AFW-CIL GỐC (Hạt Seed của tác giả):")
_, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
w_init = compute_weight(X_tr, y_tr)
best_w_author, gm_author = lfb_pso(C_val, posX, negX, w_init, T_val, X_te, y_te, X_tr, y_tr, 5, 0.1, 0.5, 0.1, 0.5)
print(f" -> G-mean AFW-CIL Tác giả (K=5, s1/s3=0.1, s2/s4=0.5): {gm_author:.4f}")

print("\n3. ĐÁNH GIÁ ĐỀ XUẤT: BL-SMOTE + ADVANCED PSO-AFW-CIL:")
# Ép SMOTE
X_tr_sm, y_tr_sm = BorderlineSMOTE(random_state=42).fit_resample(X_tr, y_tr)[:2]
bounds = [(3, 11), (0.01, 0.5), (0.01, 0.99), (0.01, 0.5), (0.01, 0.99)]

# Gọi PSO (Chạy 10 hạt, 10 vòng để test nhanh)
pso_engine = ADVANCED_PSO_AFWCIL(num_particles=10, iters=10, bounds=bounds, C=C_val, T=T_val, 
                                 X_train=X_tr_sm, y_train=y_tr_sm, X_test=X_te, y_test=y_te)
best_p, best_g = pso_engine.optimize()

print("\n" + "*"*50)
print(f"🏆 KẾT LUẬN SAU 1 FOLD:")
print(f"- Base W-SVM      : {gm_base:.4f}")
print(f"- AFW-CIL Tác giả : {gm_author:.4f}")
print(f"- ĐỀ XUẤT Lai     : {best_g:.4f}")
print("*"*50)

sys.path += /home/quangvd/project/FAIR-2022
✔ RUN_DIR: ./Experiment/Test_haberman_13032026_083026
✔ LOG_DIR: ./Experiment/Test_haberman_13032026_083026/logs

BẮT ĐẦU TEST NHANH 1 FOLD VỚI PHƯƠNG PHÁP TỐI ƯU TOÀN DIỆN
=

1. ĐÁNH GIÁ BASELINE (W-SVM):
 -> G-mean W-SVM: 0.0000

2. ĐÁNH GIÁ AFW-CIL GỐC (Hạt Seed của tác giả):
 -> G-mean AFW-CIL Tác giả (K=5, s1/s3=0.1, s2/s4=0.5): 0.0000

3. ĐÁNH GIÁ ĐỀ XUẤT: BL-SMOTE + ADVANCED PSO-AFW-CIL:
👉 Khởi tạo ban đầu | G-mean Tốt nhất (thường là Hạt Seed): 0.7258
Iteration 01/10 | w=0.90, c1=2.50, c2=0.50 | GBest: 0.7258 | Params: K=5, s1=0.10, s2=0.50, s3=0.10, s4=0.50
Iteration 02/10 | w=0.85, c1=2.30, c2=0.70 | GBest: 0.7258 | Params: K=5, s1=0.10, s2=0.50, s3=0.10, s4=0.50
Iteration 03/10 | w=0.80, c1=2.10, c2=0.90 | GBest: 0.7258 | Params: K=5, s1=0.10, s2=0.50, s3=0.10, s4=0.50
Iteration 04/10 | w=0.75, c1=1.90, c2=1.10 | GBest: 0.7258 | Params: K=5, s1=0.10, s2=0.50, s3=0.10, s4=0.50
Iteration 05/10 | w=0.70, c1=1.70, c2=1.30 | GBest: 0.72

In [4]:
# Cell 2 — ADVANCED PSO cho AFW-CIL (theo bản đặc tả)
class ADVANCED_PSO_AFWCIL:
    def __init__(self, num_particles, iters, bounds, C, T, X_train, y_train, X_test, y_test, random_state=None):
        self.num_particles = int(num_particles)
        self.iters = int(iters)
        self.bounds = list(bounds)
        self.C = C
        self.T = T
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.random_state = random_state

        # Chuẩn bị thành phần AFW-CIL
        _, self.ind_posX, self.ind_negX = is_tomek(X_train, y_train, class_type=[-1.0])
        self.init_weight = compute_weight(X_train, y_train)
        self.dim = len(self.bounds)

        # RNG cục bộ để tái lập nếu cần
        self.rng = np.random.default_rng(self.random_state)

    def _clip_particle(self, particle):
        p = np.asarray(particle, dtype=float).copy()
        for d, (low, high) in enumerate(self.bounds):
            p[d] = np.clip(p[d], low, high)
        return p

    def _decode_particle(self, particle):
        p = self._clip_particle(particle)

        # K phải là integer, ép nằm trong bounds và làm tròn
        k_low, k_high = self.bounds[0]
        K = int(np.rint(p[0]))
        K = int(np.clip(K, int(np.ceil(k_low)), int(np.floor(k_high))))

        s1 = float(np.clip(p[1], self.bounds[1][0], self.bounds[1][1]))
        s2 = float(np.clip(p[2], self.bounds[2][0], self.bounds[2][1]))
        s3 = float(np.clip(p[3], self.bounds[3][0], self.bounds[3][1]))
        s4 = float(np.clip(p[4], self.bounds[4][0], self.bounds[4][1]))
        return K, s1, s2, s3, s4

    def fitness(self, particle):
        K, s1, s2, s3, s4 = self._decode_particle(particle)
        _, gmax = lfb_pso(
            self.C, self.ind_posX, self.ind_negX, self.init_weight, self.T,
            self.X_test, self.y_test, self.X_train, self.y_train,
            K, s1, s2, s3, s4
        )
        return float(gmax)

    def optimize(self):
        dim = self.dim
        n = self.num_particles

        # ===== 1) Khởi tạo có Seeding =====
        pos = np.zeros((n, dim), dtype=float)

        # Particle 0: seed cố định theo đặc tả tác giả
        pos[0] = np.array([5.0, 0.1, 0.5, 0.1, 0.5], dtype=float)

        # Particle 1..N-1: random uniform trong bounds
        if n > 1:
            for d, (low, high) in enumerate(self.bounds):
                pos[1:, d] = self.rng.uniform(low, high, size=n - 1)

        # Clip lại để đảm bảo seed/random không vượt bounds
        for j in range(n):
            pos[j] = self._clip_particle(pos[j])

        # Vận tốc ban đầu
        vel = np.zeros_like(pos)

        # pbest / gbest ban đầu
        pbest_pos = pos.copy()
        pbest_val = np.array([self.fitness(pos[j]) for j in range(n)], dtype=float)
        gbest_idx = int(np.argmax(pbest_val))
        gbest_pos = pbest_pos[gbest_idx].copy()
        gbest_val = float(pbest_val[gbest_idx])

        # ===== 2) Hệ số LDIW + TVAC =====
        w_max, w_min = 0.9, 0.4
        c1_max, c1_min = 2.5, 0.5
        c2_max, c2_min = 2.5, 0.5

        convergence = []

        for i in range(self.iters):
            ratio = i / self.iters

            w = w_max - (w_max - w_min) * ratio
            c1 = c1_max - (c1_max - c1_min) * ratio
            c2 = c2_min + (c2_max - c2_min) * ratio

            for j in range(n):
                r1 = self.rng.random(dim)
                r2 = self.rng.random(dim)

                # Cập nhật vận tốc
                vel[j] = (
                    w * vel[j]
                    + c1 * r1 * (pbest_pos[j] - pos[j])
                    + c2 * r2 * (gbest_pos - pos[j])
                )

                # Cập nhật vị trí
                pos[j] = pos[j] + vel[j]

                # Clip bounds ngay sau update vị trí
                pos[j] = self._clip_particle(pos[j])

                # Đánh giá fitness và cập nhật best
                fval = self.fitness(pos[j])
                if fval > pbest_val[j]:
                    pbest_val[j] = fval
                    pbest_pos[j] = pos[j].copy()

                if fval > gbest_val:
                    gbest_val = fval
                    gbest_pos = pos[j].copy()

            K_best, s1_best, s2_best, s3_best, s4_best = self._decode_particle(gbest_pos)
            convergence.append(gbest_val)

            # Logging đúng format đặc tả
            print(
                f"Iteration {i+1}/{self.iters} | "
                f"w={w:.2f}, c1={c1:.2f}, c2={c2:.2f} | "
                f"GBest: {gbest_val:.4f} | "
                f"Params: K={K_best}, s1={s1_best:.3f}, s2={s2_best:.3f}, s3={s3_best:.3f}, s4={s4_best:.3f}"
            )

        return gbest_pos, gbest_val, convergence


# Gợi ý sử dụng:
# bounds = [(3, 11), (0.01, 0.5), (0.01, 0.99), (0.01, 0.5), (0.01, 0.99)]
# pso = ADVANCED_PSO_AFWCIL(10, 10, bounds, C_val, T_val, X_tr_sm, y_tr_sm, X_te, y_te, random_state=42)
# best_p, best_g, conv = pso.optimize()

In [6]:
# Cell 3 — Test nhanh 1 Fold cho class ở Cell 2 (IR=4%)
print("\n" + "="*70)
print("CELL 3 | TEST 1 FOLD với ADVANCED_PSO_AFWCIL (Cell 2) | IR=4%")
print("="*70)

# 1) Load + map nhãn
_df = pd.read_csv(DATASET_FILE)
_df['class'] = _df['class'].map({2: 1.0, 1: -1.0})
y_all_raw = np.asarray(_df['class'].to_numpy(), dtype=float)
X_all_raw = np.asarray(_df.drop(columns=['class']).to_numpy(), dtype=float)

# 2) Giảm dữ liệu về IR mục tiêu = 4%
IR_TARGET = 0.04
rng_ir = np.random.default_rng(42)

pos_idx = np.where(y_all_raw == 1.0)[0]
neg_idx = np.where(y_all_raw == -1.0)[0]

n_neg = len(neg_idx)
# Giữ toàn bộ lớp âm, tính số dương cần giữ để đạt pos/(pos+neg) ~= IR_TARGET
n_pos_target = int(round((IR_TARGET * n_neg) / (1.0 - IR_TARGET)))
n_pos_target = max(5, min(len(pos_idx), n_pos_target))  # để đủ cho 5-fold

sel_pos = rng_ir.choice(pos_idx, size=n_pos_target, replace=False)
sel_idx = np.concatenate([neg_idx, sel_pos])
rng_ir.shuffle(sel_idx)

X_all_c3 = X_all_raw[sel_idx]
y_all_c3 = y_all_raw[sel_idx]

actual_ir = float(np.sum(y_all_c3 == 1.0) / len(y_all_c3))
print(f"IR target={IR_TARGET:.4f} | IR actual={actual_ir:.4f} | total={len(y_all_c3)}")
print(f"Class counts (+1/-1): {int(np.sum(y_all_c3==1.0))}/{int(np.sum(y_all_c3==-1.0))}")

# 3) Lấy đúng 1 fold cố định
_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, te_idx = next(_skf.split(X_all_c3, y_all_c3))

_scaler = StandardScaler().fit(X_all_c3[tr_idx])
X_tr_c3 = _scaler.transform(X_all_c3[tr_idx])
X_te_c3 = _scaler.transform(X_all_c3[te_idx])
y_tr_c3 = y_all_c3[tr_idx]
y_te_c3 = y_all_c3[te_idx]

print(f"Train size: {len(y_tr_c3)} | Test size: {len(y_te_c3)}")
print(f"Train (+1/-1): {int(np.sum(y_tr_c3==1.0))}/{int(np.sum(y_tr_c3==-1.0))}")
print(f"Test  (+1/-1): {int(np.sum(y_te_c3==1.0))}/{int(np.sum(y_te_c3==-1.0))}")

# 4) Baseline W-SVM
C_c3, T_c3 = 100, 5
_baseline = Wsvm(C_c3, np.ones(len(y_tr_c3)))
_baseline.fit(X_tr_c3, y_tr_c3)
y_pred_base_c3 = _baseline.predict(X_te_c3)
sp_b, se_b, gm_b = Gmean(y_te_c3, y_pred_base_c3)
print(f"\nBaseline W-SVM | SP={sp_b:.4f} SE={se_b:.4f} Gmean={gm_b:.4f}")

# 5) AFW-CIL tác giả (seed)
_, _posX, _negX = is_tomek(X_tr_c3, y_tr_c3, class_type=[-1.0])
_w_init = compute_weight(X_tr_c3, y_tr_c3)
if np.any(np.isnan(_w_init)) or np.any(_w_init <= 0):
    _w_init = np.ones(len(y_tr_c3))

_w_author, gm_author_c3 = lfb_pso(
    C_c3, _posX, _negX, _w_init, T_c3,
    X_te_c3, y_te_c3, X_tr_c3, y_tr_c3,
    5, 0.1, 0.5, 0.1, 0.5
)
_model_author = Wsvm(C_c3, _w_author)
_model_author.fit(X_tr_c3, y_tr_c3)
y_pred_author_c3 = _model_author.predict(X_te_c3)
sp_a, se_a, gm_a = Gmean(y_te_c3, y_pred_author_c3)
print(f"AFW-CIL seed   | SP={sp_a:.4f} SE={se_a:.4f} Gmean={gm_a:.4f}")

# 6) BL-SMOTE + Advanced PSO (class ở Cell 2)
X_tr_sm_c3, y_tr_sm_c3 = BorderlineSMOTE(random_state=42).fit_resample(X_tr_c3, y_tr_c3)[:2]
_bounds_c3 = [(3, 11), (0.01, 0.5), (0.01, 0.99), (0.01, 0.5), (0.01, 0.99)]

pso_c3 = ADVANCED_PSO_AFWCIL(
    num_particles=10,
    iters=10,
    bounds=_bounds_c3,
    C=C_c3,
    T=T_c3,
    X_train=X_tr_sm_c3,
    y_train=y_tr_sm_c3,
    X_test=X_te_c3,
    y_test=y_te_c3,
    random_state=42
)

best_p_c3, best_g_c3, conv_c3 = pso_c3.optimize()
K_c3 = int(np.rint(best_p_c3[0]))

# Dự đoán lại bằng best params PSO
_, _posX2, _negX2 = is_tomek(X_tr_sm_c3, y_tr_sm_c3, class_type=[-1.0])
_w_init2 = compute_weight(X_tr_sm_c3, y_tr_sm_c3)
if np.any(np.isnan(_w_init2)) or np.any(_w_init2 <= 0):
    _w_init2 = np.ones(len(y_tr_sm_c3))

_w_best_c3, _ = lfb_pso(
    C_c3, _posX2, _negX2, _w_init2, T_c3,
    X_te_c3, y_te_c3, X_tr_sm_c3, y_tr_sm_c3,
    K_c3, best_p_c3[1], best_p_c3[2], best_p_c3[3], best_p_c3[4]
)

_model_best = Wsvm(C_c3, _w_best_c3)
_model_best.fit(X_tr_sm_c3, y_tr_sm_c3)
y_pred_best_c3 = _model_best.predict(X_te_c3)
sp_p, se_p, gm_p = Gmean(y_te_c3, y_pred_best_c3)

print("\n" + "-"*70)
print("KẾT QUẢ CELL 3 (1 FOLD, IR=4%):")
print(f"W-SVM baseline        : {gm_b:.4f}")
print(f"AFW-CIL seed (author) : {gm_a:.4f}")
print(f"Advanced PSO + SMOTE  : {gm_p:.4f} (best_g={best_g_c3:.4f})")
print(f"Best params           : K={K_c3}, s1={best_p_c3[1]:.3f}, s2={best_p_c3[2]:.3f}, s3={best_p_c3[3]:.3f}, s4={best_p_c3[4]:.3f}")
print(f"Convergence length    : {len(conv_c3)}")
print("-"*70)


CELL 3 | TEST 1 FOLD với ADVANCED_PSO_AFWCIL (Cell 2) | IR=4%
IR target=0.0400 | IR actual=0.0385 | total=234
Class counts (+1/-1): 9/225
Train size: 187 | Test size: 47
Train (+1/-1): 7/180
Test  (+1/-1): 2/45

Baseline W-SVM | SP=1.0000 SE=0.0000 Gmean=0.0000
AFW-CIL seed   | SP=1.0000 SE=0.0000 Gmean=0.0000
Iteration 1/10 | w=0.90, c1=2.50, c2=0.50 | GBest: 0.6325 | Params: K=5, s1=0.100, s2=0.500, s3=0.100, s4=0.500
Iteration 2/10 | w=0.85, c1=2.30, c2=0.70 | GBest: 0.6325 | Params: K=5, s1=0.100, s2=0.500, s3=0.100, s4=0.500
Iteration 3/10 | w=0.80, c1=2.10, c2=0.90 | GBest: 0.6325 | Params: K=5, s1=0.100, s2=0.500, s3=0.100, s4=0.500
Iteration 4/10 | w=0.75, c1=1.90, c2=1.10 | GBest: 0.6325 | Params: K=5, s1=0.100, s2=0.500, s3=0.100, s4=0.500
Iteration 5/10 | w=0.70, c1=1.70, c2=1.30 | GBest: 0.6325 | Params: K=5, s1=0.100, s2=0.500, s3=0.100, s4=0.500
Iteration 6/10 | w=0.65, c1=1.50, c2=1.50 | GBest: 0.6325 | Params: K=5, s1=0.100, s2=0.500, s3=0.100, s4=0.500
Iteration 7/10 